
# Quantum-RAG Rating Predictor (QRRP) : Amazon Books Reviews

Ce notebook implémente un pipeline **hybride LLM + RAG + Quantum** pour la prédiction de ratings (1–5 étoiles) sur les reviews Amazon Books.

Architecture globale :

1. **Dataset** : `cogsci13/Amazon-Reviews-2023-Books-Review` (Hugging Face)
2. **Embeddings** : encodeur sémantique (SentenceTransformer)
3. **Index de similarité** : `NearestNeighbors` (cosine) pour retrouver les reviews proches
4. **Quantum Retrieval** : circuit quantique (PennyLane) pour calculer des poids non linéaires sur les voisins les plus similaires
5. **RAG + LLM** : Mistral-7B-Instruct lit la review cible + les reviews similaires (avec leurs vrais ratings) et prédit un rating
6. **Fusion Hybride** : fusion du rating LLM et du rating kNN quantique pour produire un **rating final**

> ⚠️ Remarques :
> - Le LLM utilisé est `mistralai/Mistral-7B-Instruct-v0.2`, qui nécessite GPU + RAM suffisants.
> - Le module quantique est simulé via `default.qubit` (PennyLane).
> - Ce pipeline est **zero-shot** : aucun entraînement n'est effectué sur le dataset (pas de train/test split nécessaire dans le sens classique).


## 1. Imports & Installation

In [5]:
!pip install transformers accelerate sentencepiece datasets sentence-transformers pennylane scikit-learn -q

import math
import random
import json
import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

import pennylane as qml

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


You should consider upgrading via the '/faststorage/project/DEIC-SDU-L2-22/env/bin/python3 -m pip install --upgrade pip' command.


## 2. Configuration

In [ ]:
HF_DATASET_NAME = "cogsci13/Amazon-Reviews-2023-Books-Review"
HF_SPLIT_CANDIDATES = ["full", "train", "default"]

TEXT_COL = "text"
RATING_COL = "rating"

# MAX_DOCS = 1000
N_TEST = 300  # valeur arbitraire, le vrai df sera chargé après

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MAX_NEIGHBORS = 4

LLM_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 128

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


In [ ]:
def load_amazon_books_dataset():
    last_err = None
    for split in HF_SPLIT_CANDIDATES:
        try:
            print(f"🔹 Tentative de chargement avec split='{split}'...")
            ds = load_dataset(HF_DATASET_NAME, split=split)
            print("✅ Chargé avec succès :", ds)
            return ds
        except Exception as e:
            print("❌ Échec pour split", split, ":", e)
            last_err = e
    raise last_err

In [ ]:
ds = load_amazon_books_dataset()
MAX_DOCS= None
if MAX_DOCS is not None and len(ds) > MAX_DOCS:
    ds_sub = ds.shuffle(seed=RANDOM_SEED).select(range(MAX_DOCS))
else:
    ds_sub = ds

df = ds_sub.to_pandas()
df[RATING_COL] = df[RATING_COL].round().astype(int)
df = df[[TEXT_COL, RATING_COL]].dropna().reset_index(drop=True)

# Maintenant df existe, on peut définir N_TEST avec df
N_TEST = len(df)

print("DataFrame final:", df.shape)
df.head()


## 3. Loading the Amazon Books dataset (Hugging Face)

In [ ]:

# def load_amazon_books_dataset():
#     last_err = None
#     for split in HF_SPLIT_CANDIDATES:
#         try:
#             print(f"🔹 Tentative de chargement avec split='{split}'...")
#             ds = load_dataset(HF_DATASET_NAME, split=split)
#             print("✅ Chargé avec succès :", ds)
#             return ds
#         except Exception as e:
#             print("❌ Échec pour split", split, ":", e)
#             last_err = e
#     raise last_err

# ds = load_amazon_books_dataset()

# if MAX_DOCS is not None and len(ds) > MAX_DOCS:
#     ds_sub = ds.shuffle(seed=RANDOM_SEED).select(range(MAX_DOCS))
# else:
#     ds_sub = ds

# print("\nSous-ensemble utilisé :", len(ds_sub))

# df = ds_sub.to_pandas()

# if TEXT_COL not in df.columns or RATING_COL not in df.columns:
#     print("Colonnes disponibles :", df.columns)
#     raise ValueError(f"Les colonnes '{TEXT_COL}' ou '{RATING_COL}' sont introuvables dans le dataset.")

# df[RATING_COL] = df[RATING_COL].round().astype(int)
# df = df[[TEXT_COL, RATING_COL]].dropna().reset_index(drop=True)

# print("DataFrame final:", df.shape)
# df.head()


## 4. Embedding model & Similarity index

In [ ]:
# ============================================================
# 🔹 LIMITATION DU DATASET (OPTION 1 : SAMPLING)
# ============================================================
MAX_DOCS = 5000   # 🔴 AJUSTE ICI (1000 / 3000 / 5000 max recommandé)

print("📊 Dataset original :", len(df))

if len(df) > MAX_DOCS:
    df = df.sample(n=MAX_DOCS, random_state=42).reset_index(drop=True)

print("✅ Dataset utilisé pour embeddings :", len(df))


# ============================================================
# 🔹 CHARGEMENT DU MODELE D'EMBEDDING
# ============================================================
print("🔹 Chargement du modèle d'embedding :", EMBEDDING_MODEL_NAME)
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)


# ============================================================
# 🔹 ENCODAGE DES REVIEWS
# ============================================================
print("🔹 Encodage des reviews...")
corpus_texts = df[TEXT_COL].astype(str).tolist()

corpus_embeddings = embed_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True
)

corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)


# ============================================================
# 🔹 INFORMATIONS POST-ENCODAGE
# ============================================================
print("📊 Shape des embeddings :", corpus_embeddings.shape)
print("📊 Dimension embedding :", corpus_embeddings.shape[1])

mem_mb = corpus_embeddings.nbytes / (1024 ** 2)
print(f"📊 Mémoire embeddings : {mem_mb:.2f} MB")


# ============================================================
# 🔹 CONSTRUCTION DE L'INDEX NN
# ============================================================
print("🔹 Construction de l'index NearestNeighbors (cosine)...")
nn = NearestNeighbors(
    n_neighbors=MAX_NEIGHBORS,
    metric="cosine"
)
nn.fit(corpus_embeddings)

print(f"✅ Index prêt sur {corpus_embeddings.shape[0]} reviews.")


## 5. Quantum module: neighbor reweighting(Quantum Retrieval)

In [ ]:
print("Kernel check:")
print("df" in globals(), "hybrid_predict" in globals(), "MAX_NEIGHBORS" in globals())

In [ ]:
print("Nombre de reviews encodées =", len(corpus_texts))
print("Exemple review length =", len(corpus_texts[0]) if len(corpus_texts)>0 else None)


In [ ]:
MAX_NEIGHBORS = 4          # ou 3, 5, etc.

In [ ]:
n_qubits = MAX_NEIGHBORS
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_weighting_circuit(angles):
    for i in range(n_qubits):
        qml.RY(angles[i], wires=i)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

def quantum_reweight(similarities):
    sims = np.array(similarities, dtype=float)
    sims = np.clip(sims, 0.0, None)
    if sims.max() > 0:
        sims = sims / sims.max()
    angles = np.pi * sims
    z_expect = np.array(quantum_weighting_circuit(angles))
    raw = (1.0 - z_expect) / 2.0
    s = raw.sum()
    if s <= 0:
        return np.ones_like(raw) / len(raw)
    return raw / s


## 6. Fonctions de retrieval (classique + quantique)

In [ ]:

def retrieve_neighbors(review_text, top_k=MAX_NEIGHBORS):
    q_emb = embed_model.encode([review_text])[0].reshape(1, -1)
    distances, indices = nn.kneighbors(q_emb, n_neighbors=top_k)
    distances = distances[0]
    indices = indices[0]
    sims = 1.0 - distances
    neighbors = []
    for sim, idx in zip(sims, indices):
        neighbors.append({
            "index": int(idx),
            "similarity": float(sim),
            "text": df.loc[idx, TEXT_COL],
            "rating": int(df.loc[idx, RATING_COL]),
        })
    return neighbors

def quantum_knn_rating(neighbors):
    if len(neighbors) == 0:
        return 3.0
    sims = [max(0.0, n["similarity"]) for n in neighbors]
    q_weights = quantum_reweight(sims)
    ratings = np.array([n["rating"] for n in neighbors], dtype=float)
    rating_cont = float(np.dot(q_weights, ratings))
    return rating_cont


## 7. Chargement du LLM Mistral-7B-Instruct

In [ ]:

print("🔹 Chargement du tokenizer & modèle LLM :", LLM_NAME)
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
model.to(DEVICE)
print("✅ Modèle chargé sur :", DEVICE)


In [ ]:
# === Figure Grover circuit (PennyLane) -> PNG + PDF (vector) ===
import math
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

# -----------------------------
# Paramètres (IDENTIQUES à ton code)
# -----------------------------
N_RATINGS = 5
N_QUBITS_GROVER = 3   # 2^3 = 8 >= 5

def int_to_bits(x, n_bits):
    """Convertit un entier en liste de bits de longueur n_bits."""
    return [int(b) for b in format(x, f"0{n_bits}b")]

def grover_oracle(target_index):
    """
    Oracle de Grover : marque l'état target_index en appliquant une
    MultiControlledX après mapping vers |111>.
    (IDENTIQUE à ton code)
    """
    bits = int_to_bits(target_index, N_QUBITS_GROVER)

    # Amener l'état cible en |111...> via des X sur les bits à 0
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    # Revenir au basis standard
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

def grover_diffuser():
    """Diffuser standard : inversion around the mean. (IDENTIQUE à ton code)"""
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)
        qml.PauliX(wires=i)

    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    for i in range(N_QUBITS_GROVER):
        qml.PauliX(wires=i)
        qml.Hadamard(wires=i)

# -----------------------------
# QNode "dessinable"
# IMPORTANT : device analytic (shots=None) pour un rendu stable
# -----------------------------
dev_draw = qml.device("default.qubit", wires=N_QUBITS_GROVER, shots=None)

@qml.qnode(dev_draw)
def grover_circuit_draw(target_index, n_iterations):
    # Superposition uniforme (IDENTIQUE)
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)

    # Oracle + diffuser (IDENTIQUE)
    for _ in range(n_iterations):
        grover_oracle(target_index)
        grover_diffuser()

    # On met une mesure simple pour permettre le rendu du circuit
    # (ça ne change PAS les portes affichées, juste l'objet "return")
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_GROVER)]

# -----------------------------
# Génération & export
# -----------------------------
target_rating = 5                    # juste pour illustrer (1..5)
target_index  = int(target_rating) - 1
n_iterations  = 1                    # 1 ou 2 max pour garder lisible

drawer = qml.draw_mpl(grover_circuit_draw, decimals=2)
fig, ax = drawer(target_index, n_iterations)

# Style "paper-like"
fig.set_size_inches(14, 3.2)
ax.set_title("")                     # pas de titre dans la figure
fig.tight_layout()

# Exports
fig.savefig("grover_circuit.png", dpi=600, bbox_inches="tight")   # PNG net
fig.savefig("grover_circuit.pdf", bbox_inches="tight")            # PDF vectoriel (TOP pour LaTeX)
plt.close(fig)

print("✅ Saved: grover_circuit.png + grover_circuit.pdf")


In [ ]:
# === Figure Grover circuit (PennyLane) -> PNG ===
import math
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

# -----------------------------
# Paramètres (comme ton code)
# -----------------------------
N_RATINGS = 5
N_QUBITS_GROVER = 3

def int_to_bits(x, n_bits):
    return [int(b) for b in format(x, f"0{n_bits}b")]

def grover_oracle(target_index):
    bits = int_to_bits(target_index, N_QUBITS_GROVER)

    # Bring target state to |111...>
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    # Undo basis change
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

def grover_diffuser():
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)
        qml.PauliX(wires=i)

    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    for i in range(N_QUBITS_GROVER):
        qml.PauliX(wires=i)
        qml.Hadamard(wires=i)

# -----------------------------
# Circuit "dessinable"
# (device analytic pour un rendu propre)
# -----------------------------
dev_draw = qml.device("default.qubit", wires=N_QUBITS_GROVER)

@qml.qnode(dev_draw)
def grover_circuit_draw(target_index, n_iterations):
    # Uniform superposition
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)

    # Oracle + diffuser repeated
    for _ in range(n_iterations):
        grover_oracle(target_index)
        grover_diffuser()

    # Comme ton code renvoie sample, mais pour le dessin on met une mesure simple
    # (le dessin reste correct, et évite des sorties "sample" dépendantes des shots)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_GROVER)]

# -----------------------------
# Génération de la figure PNG
# -----------------------------
target_rating = 5                 # juste pour la figure (tu peux changer)
target_index = int(target_rating) - 1  # 1..5 -> 0..4
n_iterations = 1                  # pour figure lisible (tu peux mettre 2)

drawer = qml.draw_mpl(grover_circuit_draw, decimals=2)
fig, ax = drawer(target_index, n_iterations)

# Taille + export haute qualité
fig.set_size_inches(14, 3.2)      # ajuste si tu veux plus large/haut
fig.savefig("grover_circuit.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("✅ Saved: grover_circuit.png")


## 8. Prompt RAG + appel LLM (rating à partir de la review + voisins)

In [ ]:

SYSTEM_PROMPT = """You are an expert analyst of Amazon BOOK reviews.

Your task is to read a TARGET review about a BOOK and some SIMILAR REVIEWS (with their true ratings),
and infer the most likely STAR RATING (1 to 5) that the user would give to the TARGET review.

The rating must be an INTEGER between 1 and 5, where:
1 = very negative opinion
2 = negative opinion
3 = mixed / neutral opinion
4 = positive opinion
5 = very positive opinion

You MUST respond in valid JSON, and ONLY valid JSON, with the following fields:
- "rating": integer from 1 to 5
- "confidence": float between 0 and 1
- "reasoning": short text explaining why this rating was predicted

Do NOT add any text before or after the JSON. Do NOT add markdown or code blocks.
"""


def build_rag_prompt(target_review: str, neighbors):
    prompt = SYSTEM_PROMPT + "\n\n"
    prompt += "TARGET REVIEW:\n"
    prompt += target_review.strip() + "\n\n"

    if neighbors:
        prompt += "SIMILAR REVIEWS WITH THEIR TRUE RATINGS:\n"
        for i, n in enumerate(neighbors):
            snippet = n["text"].replace("\n", " ")
            if len(snippet) > 300:
                snippet = snippet[:300] + "..."
            prompt += f"- Review {i+1}: rating={n['rating']}, text=\"{snippet}\"\n"
        prompt += "\n"
    else:
        prompt += "No similar reviews were found.\n\n"

    prompt += "Based on the TARGET REVIEW and the SIMILAR REVIEWS (and their ratings), "
    prompt += "predict the most likely star rating (1 to 5) for the TARGET REVIEW. "
    prompt += "Return ONLY a JSON object with keys: rating, confidence, reasoning."
    return prompt


def call_mistral_with_rag(review_text: str, neighbors):
    prompt = build_rag_prompt(review_text, neighbors)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    start = generated.find("{")
    end = generated.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {
            "rating": 3,
            "confidence": 0.0,
            "reasoning": "no json found",
            "raw": generated,
        }

    json_str = generated[start:end+1]

    try:
        data = json.loads(json_str)
    except Exception:
        data = {
            "rating": 3,
            "confidence": 0.0,
            "reasoning": "fallback parse error",
            "raw": generated,
        }

    try:
        rating = int(data.get("rating", 3))
    except Exception:
        rating = 3
    rating = max(1, min(5, rating))
    data["rating"] = rating

    try:
        confidence = float(data.get("confidence", 0.0))
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))
    data["confidence"] = confidence

    return data


### 🔎 Test rapide du pipeline LLM+RAG sur une review

In [ ]:
sample_text = df.loc[0, TEXT_COL]
print("TARGET REVIEW :\n", sample_text[:500], "\n")

neighbors = retrieve_neighbors(sample_text, top_k=MAX_NEIGHBORS)
print("Voisins récupérés :")

for n in neighbors:
    snippet = n["text"][:80].replace("\n", " ")   # <-- correction ici
    print(f"- rating={n['rating']}, sim={n['similarity']:.3f}, text={snippet}...")

q_knn = quantum_knn_rating(neighbors)
print("\nRating quantum kNN (continu) :", q_knn)

llm_out = call_mistral_with_rag(sample_text, neighbors)
print("\nSortie LLM :")
print(llm_out)


## 9. Fonction de prédiction hybride : Quantum-RAG + LLM

In [ ]:
# without ablation study lamda
def hybrid_predict(review_text: str):
    neighbors = retrieve_neighbors(review_text, top_k=MAX_NEIGHBORS)

    q_knn_cont = quantum_knn_rating(neighbors)
    q_knn_rating = int(round(q_knn_cont))
    q_knn_rating = max(1, min(5, q_knn_rating))

    llm_out = call_mistral_with_rag(review_text, neighbors)
    llm_rating = llm_out["rating"]
    confidence = llm_out.get("confidence", 0.0)
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    base_w_llm = 0.7
    base_w_q = 0.3

    w_llm = base_w_llm + 0.2 * confidence
    w_q = base_w_q * (1.0 - confidence)

    s = w_llm + w_q
    w_llm /= s
    w_q /= s

    fused_cont = w_llm * llm_rating + w_q * q_knn_cont
    final_rating = int(round(fused_cont))
    final_rating = max(1, min(5, final_rating))

    return {
        "llm_rating": llm_rating,
        "quantum_knn_rating_cont": q_knn_cont,
        "quantum_knn_rating": q_knn_rating,
        "final_rating": final_rating,
        "llm_raw": llm_out,
        "weights": {"w_llm": w_llm, "w_q": w_q},
        "neighbors": neighbors,
    }


In [ ]:
def hybrid_predict(review_text: str, lam: float = 0.5):
    neighbors = retrieve_neighbors(review_text, top_k=MAX_NEIGHBORS)

    q_knn_cont = quantum_knn_rating(neighbors)
    q_knn_rating = int(round(q_knn_cont))
    q_knn_rating = max(1, min(5, q_knn_rating))

    llm_out = call_mistral_with_rag(review_text, neighbors)
    llm_rating = llm_out["rating"]

    # λ = weight of LLM
    w_llm = lam
    w_q = 1.0 - lam

    fused_cont = w_llm * llm_rating + w_q * q_knn_cont
    final_rating = int(round(fused_cont))
    final_rating = max(1, min(5, final_rating))

    return {
        "llm_rating": llm_rating,
        "quantum_knn_rating_cont": q_knn_cont,
        "quantum_knn_rating": q_knn_rating,
        "final_rating": final_rating,
        "llm_raw": llm_out,
        "weights": {"w_llm": w_llm, "w_q": w_q},
        "neighbors": neighbors,
    }


In [ ]:
import math

# On veut chercher dans un espace de 5 ratings (1..5) -> 3 qubits (2^3 = 8 >= 5)
N_RATINGS = 5
N_QUBITS_GROVER = 3
N_SHOTS_GROVER = 200

dev_grover = qml.device("default.qubit", wires=N_QUBITS_GROVER, shots=N_SHOTS_GROVER)

def int_to_bits(x, n_bits):
    """Convertit un entier en liste de bits de longueur n_bits."""
    return [int(b) for b in format(x, f"0{n_bits}b")]

def grover_oracle(target_index):
    """
    Oracle de Grover : marque l'état 'target_index'
    en appliquant une multi-controlled X (approx de multi-controlled Z).
    """
    bits = int_to_bits(target_index, N_QUBITS_GROVER)

    # Amener l'état cible en |111...> via des X sur les bits à 0
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

    # Multi-controlled X sur le dernier qubit, contrôlé par les autres
    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    # Revenir au basis standard
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

def grover_diffuser():
    """Diffuser standard : inversion around the mean."""
    # H puis X sur tous les qubits
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)
        qml.PauliX(wires=i)

    # Multi-controlled X sur le dernier qubit
    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    # X puis H sur tous les qubits
    for i in range(N_QUBITS_GROVER):
        qml.PauliX(wires=i)
        qml.Hadamard(wires=i)

def grover_iterations(N, M=1):
    """Nombre d'itérations de Grover pour N états et M solutions."""
    return max(1, round((math.pi / 4.0) * math.sqrt(N / M)))

@qml.qnode(dev_grover)
def grover_circuit(target_index, n_iterations):
    """
    Circuit de Grover :
    - superposition uniforme sur 3 qubits
    - application de l'oracle + diffuseur k fois
    - échantillonnage de la base informatique
    """
    # Superposition uniforme
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)

    for _ in range(n_iterations):
        grover_oracle(target_index)
        grover_diffuser()

    return qml.sample(wires=range(N_QUBITS_GROVER))

# def run_grover_for_rating(prior_rating: int):
#     """
#     Utilise Grover pour amplifier un rating cible (1..5).
#     prior_rating : rating entier (1..5), par ex. LLM ou kNN
#     Retourne un rating entier (1..5).
#     """
#     # Domain 1..5 -> indices 0..4
#     target_index = int(prior_rating) - 1
#     target_index = max(0, min(N_RATINGS-1, target_index))

#     N = 2 ** N_QUBITS_GROVER
#     M = 1
#     k = grover_iterations(N, M)
#     k = 1

#     samples = grover_circuit(target_index, k)
#     samples = np.array(samples)

#     indices = []
#     for row in samples:
#         v = 0
#         for b in row:
#             v = (v << 1) | int(b)
#         indices.append(v)

#     indices = np.array(indices)
#     # On garde seulement les 0..4 (ratings valides)
#     indices = indices[indices < N_RATINGS]
#     if len(indices) == 0:
#         return int(prior_rating)

#     values, counts = np.unique(indices, return_counts=True)
#     best_index = values[np.argmax(counts)]
#     quantum_rating = int(best_index) + 1  # 0..4 -> 1..5

#     return quantum_rating


In [ ]:
def run_grover_for_rating(prior_rating: int):
    """
    Utilise Grover pour amplifier un rating cible (1..5) avec une version 'adoucie':
    - 1 seule itération (k=1)
    - clamp des indices au domaine [0..4]
    """
    # Domain 1..5 -> indices 0..4
    target_index = int(prior_rating) - 1
    target_index = max(0, min(N_RATINGS - 1, target_index))

    N = 2 ** N_QUBITS_GROVER
    # k = grover_iterations(N, M=1)   # <-- on n'utilise plus la formule théorique
    k = 1  # ✅ une seule itération pour limiter l'effet destructeur

    samples = grover_circuit(target_index, k)
    samples = np.array(samples)

    # On convertit chaque bitstring en entier
    indices = []
    for row in samples:
        v = 0
        for b in row:
            v = (v << 1) | int(b)
        indices.append(v)

    indices = np.array(indices)

    # ✅ Version améliorée : on CLAMP tout aux indices valides [0..4]
    indices_clamped = np.clip(indices, 0, N_RATINGS - 1)

    values, counts = np.unique(indices_clamped, return_counts=True)
    best_index = values[np.argmax(counts)]
    quantum_rating = int(best_index) + 1  # 0..4 -> 1..5

    return quantum_rating


## 10. Évaluation sur un sous-ensemble du dataset

In [ ]:
# N_TEST = min(N_TEST, len(df))
print("len(df) =", len(df))
print("N_TEST (avant sample) =", N_TEST)

N_TEST = len(df)
indices = random.sample(range(len(df)), N_TEST)



true_ratings = []
llm_ratings = []
qknn_ratings = []
final_ratings = []

grover_ratings = []



for idx in indices:
    review = df.loc[idx, TEXT_COL]
    y_true = int(df.loc[idx, RATING_COL])

    out = hybrid_predict(review)

    true_ratings.append(y_true)
    llm_ratings.append(out["llm_rating"])
    qknn_ratings.append(out["quantum_knn_rating"])
    final_ratings.append(out["final_rating"])
    #  prédiction Grover basée sur le rating LLM
    g_rating = run_grover_for_rating(out["llm_rating"])
    grover_ratings.append(g_rating)

true_ratings = np.array(true_ratings)
llm_ratings = np.array(llm_ratings)
qknn_ratings = np.array(qknn_ratings)
final_ratings = np.array(final_ratings)
grover_ratings = np.array(grover_ratings)

def print_metrics(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    acc = accuracy_score(y_true, y_pred)
    print(f"{name}: Accuracy={acc:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")

print("=== Résultats sur", N_TEST, "reviews ===")
print_metrics("LLM only        ", true_ratings, llm_ratings)
print_metrics("Quantum kNN only", true_ratings, qknn_ratings)
print_metrics("Hybrid final    ", true_ratings, final_ratings)
print_metrics("Grover-only LLM  ", true_ratings, grover_ratings)



## 11. Exemple détaillé d'une prédiction hybride

In [ ]:
["LLM", "Q-LLM", "Quantum kNN", "Grover-only LLM"]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    confusion_matrix,
)

# Style global (optionnel)
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 14,
})


@dataclass
class ModelResult:
    name: str
    y_true: np.ndarray
    y_pred: np.ndarray


class RatingPredictionAnalyzer:
    """
    Analyseur pour la prédiction de ratings 1–5 :
    - calcule les métriques (Accuracy, MAE, RMSE)
    - génère des figures comparant les modèles
    """

    def __init__(self, models: List[ModelResult]):
        self.models = models
        # On suppose que tous les modèles ont les mêmes y_true
        self.y_true = models[0].y_true
        self.labels = np.arange(1, 6)  # ratings 1..5

    def compute_metrics(self) -> pd.DataFrame:
        rows = []
        for m in self.models:
            mae = mean_absolute_error(m.y_true, m.y_pred)
            rmse = np.sqrt(mean_squared_error(m.y_true, m.y_pred))
            acc = accuracy_score(m.y_true, m.y_pred)
            rows.append({
                "Model": m.name,
                "Accuracy": acc,
                "MAE": mae,
                "RMSE": rmse,
            })
        df_metrics = pd.DataFrame(rows).set_index("Model")
        return df_metrics

    def plot_error_histograms(self):
        """Histogrammes des erreurs (y_pred - y_true) pour chaque modèle"""
        plt.figure(figsize=(10, 6))
        for m in self.models:
            errors = m.y_pred - m.y_true
            sns.histplot(errors, kde=True, stat="density", label=m.name, bins=np.arange(-4.5, 4.6, 1))
        plt.xlabel("Error = y_pred - y_true")
        plt.ylabel("Density")
        plt.title("Error Distribution per Model")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_abs_error_boxplot(self):
        """Boxplot des erreurs absolues par modèle"""
        data = []
        for m in self.models:
            abs_err = np.abs(m.y_pred - m.y_true)
            data.append(pd.DataFrame({
                "Model": m.name,
                "Absolute Error": abs_err
            }))
        df = pd.concat(data, ignore_index=True)

        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df, x="Model", y="Absolute Error")
        plt.title("Absolute Error Distribution per Model")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    def plot_true_vs_pred_scatter(self):
        """Scatter plot true vs predicted pour LLM et Hybrid (les deux principaux)"""
        # On cherche les modèles par nom
        name_to_model = {m.name: m for m in self.models}
        main_names = [n for n in ["LLM", "Hybrid", "Quantum kNN", "Grover-only LLM"] if n in name_to_model]

        plt.figure(figsize=(10, 6))
        for name in main_names:
            m = name_to_model[name]
            plt.scatter(m.y_true, m.y_pred, alpha=0.5, s=40, label=name)

        # diagonale parfaite
        plt.plot([1, 5], [1, 5], "k--", alpha=0.7)
        plt.xlim(0.5, 5.5)
        plt.ylim(0.5, 5.5)
        plt.xticks([1, 2, 3, 4, 5])
        plt.yticks([1, 2, 3, 4, 5])
        plt.xlabel("True Rating")
        plt.ylabel("Predicted Rating")
        plt.title("True vs Predicted Ratings")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_confusion_matrices(self):
        """Matrice de confusion pour LLM et Hybrid"""
        name_to_model = {m.name: m for m in self.models}
        selected = [n for n in ["LLM", "Hybrid"] if n in name_to_model]

        n_sel = len(selected)
        if n_sel == 0:
            print("No LLM/Hybrid models found for confusion matrices.")
            return

        fig, axes = plt.subplots(1, n_sel, figsize=(6 * n_sel, 5))

        if n_sel == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            m = name_to_model[name]
            cm = confusion_matrix(m.y_true, m.y_pred, labels=self.labels)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=self.labels, yticklabels=self.labels, ax=ax)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_title(f"Confusion Matrix - {name}")

        plt.tight_layout()
        plt.show()

    def plot_metrics_bar(self):
        """Bar chart comparant Accuracy / MAE / RMSE entre modèles"""
        df = self.compute_metrics().reset_index()

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        sns.barplot(data=df, x="Model", y="Accuracy", ax=axes[0])
        axes[0].set_title("Accuracy per Model")
        axes[0].set_ylim(0, 1)
        axes[0].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="MAE", ax=axes[1])
        axes[1].set_title("MAE per Model")
        axes[1].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="RMSE", ax=axes[2])
        axes[2].set_title("RMSE per Model")
        axes[2].grid(alpha=0.3, axis="y")

        for ax in axes:
            for label in ax.get_xticklabels():
                label.set_rotation(20)
                label.set_ha("right")

        plt.tight_layout()
        plt.show()

    def plot_quantum_advantage_box(self, baseline_name="LLM", target_name="Hybrid"):
        """
        Boîte à moustache de 'quantum advantage' en termes d'erreur absolue :
        advantage = |err_baseline| - |err_target|
        > 0 => le modèle cible (target_name) est meilleur.
        """
        name_to_model = {m.name: m for m in self.models}
        if baseline_name not in name_to_model or target_name not in name_to_model:
            print(f"Missing {baseline_name} or {target_name} for advantage plot.")
            return

        m_base = name_to_model[baseline_name]
        m_tgt = name_to_model[target_name]

        base_abs = np.abs(m_base.y_pred - m_base.y_true)
        tgt_abs = np.abs(m_tgt.y_pred - m_tgt.y_true)
        advantage = base_abs - tgt_abs  # >0 => Hybrid better

        plt.figure(figsize=(6, 5))
        sns.boxplot(data=pd.DataFrame({"Quantum Advantage": advantage}))
        plt.axhline(0.0, color="red", linestyle="--", alpha=0.7)
        plt.title(f"Quantum Advantage: |err_{baseline_name}| - |err_{target_name}|")
        plt.ylabel("Advantage (>0 means target is better)")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    def plot_all(self):
        """Pipeline rapide pour tout tracer"""
        print("=== Metrics Table ===")
        display(self.compute_metrics())  # si Jupyter, sinon print(...)
        self.plot_metrics_bar()
        self.plot_true_vs_pred_scatter()
        self.plot_confusion_matrices()
        self.plot_error_histograms()
        self.plot_abs_error_boxplot()
        self.plot_quantum_advantage_box(baseline_name="LLM", target_name="Hybrid")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Optional
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    confusion_matrix,
)

# ----------------------------
# Style global "paper-ready"
# ----------------------------
plt.style.use("default")
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 14,
})

# Palette fixe (modifie si tu veux)
MODEL_COLORS = {
    "LLM": "#9aa0a6",            # gris
    "Quantum kNN": "#f4a261",    # orange
    "Q-LLM": "#264653",          # bleu foncé (highlight)
    "Grover-only LLM": "#e63946" # rouge
}


@dataclass
class ModelResult:
    name: str
    y_true: np.ndarray
    y_pred: np.ndarray


class RatingPredictionAnalyzer:
    """
    Analyseur de prédiction de ratings (ordinal) :
    - calcule Accuracy, MAE, RMSE
    - génère des figures comparant les modèles
    """

    def __init__(self, models: List[ModelResult], labels: Optional[np.ndarray] = None):
        if not models:
            raise ValueError("models is empty")

        self.models = models

        # cast + checks
        for m in self.models:
            m.y_true = np.asarray(m.y_true).reshape(-1)
            m.y_pred = np.asarray(m.y_pred).reshape(-1)
            if len(m.y_true) != len(m.y_pred):
                raise ValueError(f"Length mismatch for model {m.name}: y_true={len(m.y_true)} vs y_pred={len(m.y_pred)}")

        # On suppose que tous les modèles ont le même y_true
        self.y_true = self.models[0].y_true

        # Labels (auto si non fournis)
        if labels is None:
            all_vals = np.concatenate([self.y_true] + [m.y_pred for m in self.models])
            # garde uniquement les valeurs finies
            all_vals = all_vals[np.isfinite(all_vals)]
            uniq = np.unique(all_vals.astype(int))
            self.labels = np.sort(uniq)
        else:
            self.labels = np.asarray(labels)

    def _color(self, name: str) -> str:
        return MODEL_COLORS.get(name, "#333333")

    def compute_metrics(self) -> pd.DataFrame:
        rows = []
        for m in self.models:
            mae = mean_absolute_error(m.y_true, m.y_pred)
            rmse = float(np.sqrt(mean_squared_error(m.y_true, m.y_pred)))
            acc = accuracy_score(m.y_true.astype(int), m.y_pred.astype(int))
            rows.append({
                "Model": m.name,
                "Accuracy": acc,
                "MAE": mae,
                "RMSE": rmse,
            })
        return pd.DataFrame(rows).set_index("Model")

    def plot_metrics_bar(self, order: Optional[List[str]] = None):
        df = self.compute_metrics().reset_index()

        # ordre: Q-LLM au centre si possible
        if order is None:
            preferred = ["LLM", "Quantum kNN", "Q-LLM", "Grover-only LLM"]
            order = [m for m in preferred if m in df["Model"].tolist()] + \
                    [m for m in df["Model"].tolist() if m not in preferred]

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        sns.barplot(data=df, x="Model", y="Accuracy", ax=axes[0],
                    order=order, palette={k: self._color(k) for k in order})
        axes[0].set_title("Accuracy")
        axes[0].set_ylim(0, 1)
        axes[0].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="MAE", ax=axes[1],
                    order=order, palette={k: self._color(k) for k in order})
        axes[1].set_title("MAE")
        axes[1].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="RMSE", ax=axes[2],
                    order=order, palette={k: self._color(k) for k in order})
        axes[2].set_title("RMSE")
        axes[2].grid(alpha=0.3, axis="y")

        for ax in axes:
            for label in ax.get_xticklabels():
                label.set_rotation(20)
                label.set_ha("right")

        plt.tight_layout()
        plt.show()

    def plot_true_vs_pred_scatter(self, names: Optional[List[str]] = None):
        name_to_model = {m.name: m for m in self.models}
        if names is None:
            # on privilégie ces modèles si présents
            preferred = ["LLM", "Q-LLM", "Quantum kNN", "Grover-only LLM"]
            names = [n for n in preferred if n in name_to_model]

        plt.figure(figsize=(10, 6))
        for name in names:
            m = name_to_model[name]
            is_main = (name == "Q-LLM")
            plt.scatter(
                m.y_true, m.y_pred,
                alpha=0.65 if not is_main else 0.9,
                s=35 if not is_main else 55,
                label=name,
                color=self._color(name),
                edgecolors="black" if is_main else "none",
                linewidths=0.7 if is_main else 0.0
            )

        # diagonale parfaite
        lo = float(min(self.labels)) - 0.5
        hi = float(max(self.labels)) + 0.5
        plt.plot([lo, hi], [lo, hi], "k--", alpha=0.7)

        plt.xlim(lo, hi)
        plt.ylim(lo, hi)
        plt.xticks(self.labels)
        plt.yticks(self.labels)
        plt.xlabel("True Rating")
        plt.ylabel("Predicted Rating")
        plt.title("True vs Predicted Ratings")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_confusion_matrices(self, selected: Optional[List[str]] = None):
        name_to_model = {m.name: m for m in self.models}
        if selected is None:
            selected = [n for n in ["LLM", "Q-LLM"] if n in name_to_model]

        if not selected:
            print("No selected models found for confusion matrices.")
            return

        n_sel = len(selected)
        fig, axes = plt.subplots(1, n_sel, figsize=(6 * n_sel, 5))
        if n_sel == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            m = name_to_model[name]
            cm = confusion_matrix(
                m.y_true.astype(int),
                m.y_pred.astype(int),
                labels=self.labels.astype(int)
            )
            sns.heatmap(
                cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=self.labels, yticklabels=self.labels, ax=ax
            )
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_title(f"Confusion Matrix — {name}")

        plt.tight_layout()
        plt.show()

    def plot_error_histograms(self, names: Optional[List[str]] = None):
        if names is None:
            names = [m.name for m in self.models]

        plt.figure(figsize=(10, 6))
        for name in names:
            m = next(mm for mm in self.models if mm.name == name)
            errors = m.y_pred - m.y_true
            sns.histplot(
                errors,
                bins=np.arange(-4.5, 4.6, 1),
                stat="density",
                element="step",
                fill=False,
                linewidth=2 if name == "Q-LLM" else 1.5,
                color=self._color(name),
                label=name
            )
        plt.xlabel("Error = y_pred − y_true")
        plt.ylabel("Density")
        plt.title("Error Distribution")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_abs_error_boxplot(self, order: Optional[List[str]] = None):
        data = []
        for m in self.models:
            abs_err = np.abs(m.y_pred - m.y_true)
            data.append(pd.DataFrame({"Model": m.name, "Absolute Error": abs_err}))
        df = pd.concat(data, ignore_index=True)

        if order is None:
            preferred = ["LLM", "Quantum kNN", "Q-LLM", "Grover-only LLM"]
            order = [m for m in preferred if m in df["Model"].unique()] + \
                    [m for m in df["Model"].unique() if m not in preferred]

        plt.figure(figsize=(8, 6))
        sns.boxplot(
            data=df, x="Model", y="Absolute Error",
            order=order,
            palette={k: self._color(k) for k in order}
        )
        plt.title("Absolute Error Distribution")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    def plot_quantum_advantage_box(self, baseline_name="LLM", target_name="Q-LLM"):
        name_to_model = {m.name: m for m in self.models}
        if baseline_name not in name_to_model or target_name not in name_to_model:
            print(f"Missing {baseline_name} or {target_name} for advantage plot.")
            return

        m_base = name_to_model[baseline_name]
        m_tgt = name_to_model[target_name]

        base_abs = np.abs(m_base.y_pred - m_base.y_true)
        tgt_abs = np.abs(m_tgt.y_pred - m_tgt.y_true)
        advantage = base_abs - tgt_abs  # >0 => target better

        plt.figure(figsize=(6, 5))
        sns.boxplot(
            data=pd.DataFrame({"Quantum Advantage": advantage}),
            color=self._color(target_name)
        )
        plt.axhline(0.0, color="red", linestyle="--", alpha=0.7)
        plt.title(f"Quantum Advantage: |err_{baseline_name}| − |err_{target_name}|")
        plt.ylabel("Advantage (>0 means Q-LLM is better)")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    def plot_all(self):
        print("=== Metrics Table ===")
        print(self.compute_metrics())  # plus robuste que display(...)
        self.plot_metrics_bar()
        self.plot_true_vs_pred_scatter()
        self.plot_confusion_matrices()
        self.plot_error_histograms()
        self.plot_abs_error_boxplot()
        self.plot_quantum_advantage_box(baseline_name="LLM", target_name="Q-LLM")


In [ ]:
models = [
    ModelResult("LLM", true_ratings, llm_ratings),
    ModelResult("Quantum kNN", true_ratings, qknn_ratings),
    ModelResult("Q-LLM", true_ratings, final_ratings),
    ModelResult("Grover-only LLM", true_ratings, grover_ratings),
]

analyzer = RatingPredictionAnalyzer(models)
analyzer.plot_all()


In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt

class RatingPredictionAnalyzer:
    # ... garde tout ton code existant ...

    def _savefig(self, save_dir, filename, dpi=600):
        if save_dir is None:
            return
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        out = os.path.join(save_dir, filename)
        plt.savefig(out, dpi=dpi, bbox_inches="tight")
        print("✅ Saved:", out)

    def plot_error_histograms(self, save_dir=None):
        plt.figure(figsize=(10, 6))
        for m in self.models:
            errors = m.y_pred - m.y_true
            sns.histplot(errors, kde=True, stat="density", label=m.name, bins=np.arange(-4.5, 4.6, 1))
        plt.xlabel("Error = y_pred - y_true")
        plt.ylabel("Density")
        plt.title("Q-LLM — Error Distribution per Model")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        self._savefig(save_dir, "01_error_histograms.png")
        plt.show()

    def plot_abs_error_boxplot(self, save_dir=None):
        data = []
        for m in self.models:
            abs_err = np.abs(m.y_pred - m.y_true)
            data.append(pd.DataFrame({"Model": m.name, "Absolute Error": abs_err}))
        df = pd.concat(data, ignore_index=True)

        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df, x="Model", y="Absolute Error")
        plt.title("Q-LLM — Absolute Error Distribution per Model")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        self._savefig(save_dir, "02_abs_error_boxplot.png")
        plt.show()

    def plot_true_vs_pred_scatter(self, save_dir=None):
        name_to_model = {m.name: m for m in self.models}
        main_names = [n for n in ["LLM", "Q-LLM", "Quantum kNN", "Grover-only LLM"] if n in name_to_model]

        plt.figure(figsize=(10, 6))
        for name in main_names:
            m = name_to_model[name]
            plt.scatter(m.y_true, m.y_pred, alpha=0.5, s=40, label=name)

        plt.plot([1, 5], [1, 5], "k--", alpha=0.7)
        plt.xlim(0.5, 5.5)
        plt.ylim(0.5, 5.5)
        plt.xticks([1, 2, 3, 4, 5])
        plt.yticks([1, 2, 3, 4, 5])
        plt.xlabel("True Rating")
        plt.ylabel("Predicted Rating")
        plt.title("Q-LLM — True vs Predicted Ratings")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        self._savefig(save_dir, "03_true_vs_pred_scatter.png")
        plt.show()

    def plot_confusion_matrices(self, save_dir=None):
        name_to_model = {m.name: m for m in self.models}
        selected = [n for n in ["LLM", "Q-LLM"] if n in name_to_model]

        if len(selected) == 0:
            print("No LLM/Q-LLM models found for confusion matrices.")
            return

        fig, axes = plt.subplots(1, len(selected), figsize=(6 * len(selected), 5))
        if len(selected) == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            m = name_to_model[name]
            cm = confusion_matrix(m.y_true, m.y_pred, labels=self.labels)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=self.labels, yticklabels=self.labels, ax=ax)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_title(f"Confusion Matrix — {name}")

        plt.tight_layout()
        self._savefig(save_dir, "04_confusion_matrices.png")
        plt.show()

    def plot_metrics_bar(self, save_dir=None):
        df = self.compute_metrics().reset_index()
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        sns.barplot(data=df, x="Model", y="Accuracy", ax=axes[0])
        axes[0].set_title("Accuracy")
        axes[0].set_ylim(0, 1)
        axes[0].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="MAE", ax=axes[1])
        axes[1].set_title("MAE")
        axes[1].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="RMSE", ax=axes[2])
        axes[2].set_title("RMSE")
        axes[2].grid(alpha=0.3, axis="y")

        for ax in axes:
            for label in ax.get_xticklabels():
                label.set_rotation(20)
                label.set_ha("right")

        plt.tight_layout()
        self._savefig(save_dir, "00_metrics_bar.png")
        plt.show()

    def plot_quantum_advantage_box(self, baseline_name="LLM", target_name="Q-LLM", save_dir=None):
        name_to_model = {m.name: m for m in self.models}
        if baseline_name not in name_to_model or target_name not in name_to_model:
            print(f"Missing {baseline_name} or {target_name} for advantage plot.")
            return

        m_base = name_to_model[baseline_name]
        m_tgt = name_to_model[target_name]
        advantage = np.abs(m_base.y_pred - m_base.y_true) - np.abs(m_tgt.y_pred - m_tgt.y_true)

        plt.figure(figsize=(6, 5))
        sns.boxplot(data=pd.DataFrame({"Quantum Advantage": advantage}))
        plt.axhline(0.0, color="red", linestyle="--", alpha=0.7)
        plt.title(f"Q-LLM — Advantage: |err_{baseline_name}| - |err_{target_name}|")
        plt.ylabel("Advantage (>0 means Q-LLM better)")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        self._savefig(save_dir, "05_quantum_advantage.png")
        plt.show()

    def plot_all(self, save_dir=None):
        print("=== Metrics Table ===")
        display(self.compute_metrics())
        self.plot_metrics_bar(save_dir=save_dir)
        self.plot_true_vs_pred_scatter(save_dir=save_dir)
        self.plot_confusion_matrices(save_dir=save_dir)
        self.plot_error_histograms(save_dir=save_dir)
        self.plot_abs_error_boxplot(save_dir=save_dir)
        self.plot_quantum_advantage_box(baseline_name="LLM", target_name="Q-LLM", save_dir=save_dir)


In [ ]:
models = [
    ModelResult("LLM",            true_ratings, llm_ratings),
    ModelResult("Quantum kNN",    true_ratings, qknn_ratings),
    ModelResult("Hybrid",         true_ratings, final_ratings),
    ModelResult("Grover-only LLM", true_ratings, grover_ratings),
]

analyzer = RatingPredictionAnalyzer(models)
analyzer.plot_all()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================
# Helpers
# =========================
def rmse(y, yhat):
    return float(np.sqrt(mean_squared_error(y, yhat)))

def acc_at_k(y, yhat, k):
    return float(np.mean(np.abs(yhat - y) <= k))

# =========================
# Figures
# =========================
def plot_accuracy_tolerance(y_true, preds_dict, ks=(0, 1, 2)):
    plt.figure(figsize=(7, 4))
    for name, yhat in preds_dict.items():
        vals = [acc_at_k(y_true, yhat, k) for k in ks]
        plt.plot(list(ks), vals, marker="o", label=name)
    plt.xticks(list(ks), [f"@{k}" for k in ks])
    plt.ylim(0, 1)
    plt.xlabel("Tolerance (|pred-true| ≤ k)")
    plt.ylabel("Accuracy")
    plt.title("Accuracy with Ordinal Tolerance")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_rating_distributions(y_true, preds_dict, labels=None):
    y_true = np.asarray(y_true).astype(int)

    if labels is None:
        labels = np.arange(int(np.min(y_true)), int(np.max(y_true)) + 1)

    plt.figure(figsize=(9, 4))
    # True distribution
    true_counts = [np.mean(y_true == l) for l in labels]
    plt.plot(labels, true_counts, marker="o", linewidth=2, label="True")

    # Pred distributions
    for name, yhat in preds_dict.items():
        yhat = np.asarray(yhat).astype(int)
        counts = [np.mean(yhat == l) for l in labels]
        plt.plot(labels, counts, marker="o", linestyle="--", label=name)

    plt.xticks(labels)
    plt.xlabel("Rating")
    plt.ylabel("Proportion")
    plt.title("Rating Distributions (True vs Predictions)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_per_class_mae(y_true, preds_dict, labels=None):
    y_true = np.asarray(y_true).astype(int)

    if labels is None:
        labels = np.arange(int(np.min(y_true)), int(np.max(y_true)) + 1)

    rows = []
    for r in labels:
        mask = (y_true == r)
        if mask.sum() == 0:
            continue
        for name, yhat in preds_dict.items():
            yhat = np.asarray(yhat).astype(int)
            rows.append({
                "True Rating": int(r),
                "Model": name,
                "MAE": mean_absolute_error(y_true[mask], yhat[mask])
            })

    df = pd.DataFrame(rows)
    pivot = df.pivot(index="True Rating", columns="Model", values="MAE")

    ax = pivot.plot(kind="bar", figsize=(10, 4))
    ax.set_xlabel("True Rating")
    ax.set_ylabel("MAE (within class)")
    ax.set_title("Per-class MAE")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

# =========================
# Build y vectors robustly
# =========================
def build_vectors_from_results(df_results: pd.DataFrame,
                              true_col="true_rating",
                              llm_col="llm_pred",
                              q_col="q_pred",
                              final_col="pred_rating"):
    missing = [c for c in [true_col, llm_col, q_col, final_col] if c not in df_results.columns]
    if missing:
        raise ValueError(
            f"Colonnes manquantes: {missing}\n"
            f"Colonnes disponibles: {list(df_results.columns)}\n"
            f"👉 Corrige les noms de colonnes dans build_vectors_from_results(...)."
        )

    y_true  = df_results[true_col].to_numpy()
    y_llm   = df_results[llm_col].to_numpy()
    y_q     = df_results[q_col].to_numpy()
    y_final = df_results[final_col].to_numpy()

    # cast + sécurité
    y_true  = np.asarray(y_true).astype(float)
    y_llm   = np.asarray(y_llm).astype(float)
    y_q     = np.asarray(y_q).astype(float)
    y_final = np.asarray(y_final).astype(float)

    # si jamais certains sont continus: arrondi + clamp 1..5
    def to_int_1_5(x):
        x = np.rint(x).astype(int)
        return np.clip(x, 1, 5)

    y_true_i  = to_int_1_5(y_true)
    y_llm_i   = to_int_1_5(y_llm)
    y_q_i     = to_int_1_5(y_q)
    y_final_i = to_int_1_5(y_final)

    return y_true_i, y_llm_i, y_q_i, y_final_i

# =========================
# MAIN: call this after evaluation
# =========================
def run_new_figures(df_results=None, partial_results=None):
    # 1) récupérer df_results
    if df_results is None:
        if partial_results is None:
            raise ValueError("Donne soit df_results, soit partial_results.")
        df_results = pd.DataFrame(partial_results)

    print("✅ df_results shape:", df_results.shape)
    print("✅ df_results columns:", list(df_results.columns))

    # 2) construire les vecteurs
    y_true, y_llm, y_q, y_final = build_vectors_from_results(df_results)

    # 3) dictionnaire modèles
    preds = {
        "LLM": y_llm,
        "Quantum": y_q,
        "Q-LLM": y_final,
    }

    # 4) plots
    plot_accuracy_tolerance(y_true, preds, ks=(0, 1, 2))
    plot_rating_distributions(y_true, preds, labels=np.arange(1, 6))
    plot_per_class_mae(y_true, preds, labels=np.arange(1, 6))

    # 5) metrics rapides
    print("\n=== Quick metrics ===")
    for name, yhat in preds.items():
        print(f"{name:8s} | Acc={np.mean(yhat==y_true):.3f} | MAE={mean_absolute_error(y_true,yhat):.3f} | RMSE={rmse(y_true,yhat):.3f}")

# =========================
# USAGE (choisis une option)
# =========================

# Option A: si tu as df_results déjà
# run_new_figures(df_results=df_results)

# Option B: si tu as partial_results (liste de dicts)
# run_new_figures(partial_results=partial_results)


In [ ]:
print("partial_results" in globals())

In [ ]:
y_true  = true_ratings.astype(int)
y_llm   = llm_ratings.astype(int)
y_q     = qknn_ratings.astype(int)
y_final = final_ratings.astype(int)

preds = {
    "LLM": y_llm,
    "Quantum": y_q,
    "Q-LLM": y_final,
}

plot_accuracy_tolerance(y_true, preds)
plot_rating_distributions(y_true, preds, labels=np.arange(1,6))
plot_per_class_mae(y_true, preds, labels=np.arange(1,6))


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass
from typing import List, Optional
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    confusion_matrix,
)

# ----------------------------
# Style (modifiable)
# ----------------------------
plt.style.use("default")
sns.set_context("talk")
sns.set_style("whitegrid")
sns.set_palette("tab10")  # palette plus "clean" que husl

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
})

# Fallback display si pas dans Jupyter
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ----------------------------
# Data structure
# ----------------------------
@dataclass
class ModelResult:
    name: str
    y_true: np.ndarray
    y_pred: np.ndarray


class RatingPredictionAnalyzer:
    """
    Analyseur pour rating prediction 1–5 :
    - calcule métriques (Accuracy, MAE, RMSE)
    - génère figures
    - sauvegarde figures en PNG/PDF
    """

    def __init__(self, models: List[ModelResult]):
        if not models:
            raise ValueError("models list is empty.")
        self.models = models

        # Vérif y_true cohérents
        base_true = np.asarray(models[0].y_true).astype(int)
        for m in models:
            if len(m.y_true) != len(base_true):
                raise ValueError(f"y_true length mismatch for model {m.name}")
        self.y_true = base_true
        self.labels = np.arange(1, 6)

    # ---------- Save helper ----------
    def _savefig(self, save_dir: Optional[str], filename: str, dpi: int = 600, also_pdf: bool = False):
        if save_dir is None:
            return
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        out_png = os.path.join(save_dir, filename)
        plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
        print("✅ Saved:", out_png)

        if also_pdf:
            out_pdf = os.path.splitext(out_png)[0] + ".pdf"
            plt.savefig(out_pdf, bbox_inches="tight")
            print("✅ Saved:", out_pdf)

    # ---------- Metrics ----------
    def compute_metrics(self) -> pd.DataFrame:
        rows = []
        for m in self.models:
            y_true = np.asarray(m.y_true).astype(int)
            y_pred = np.asarray(m.y_pred).astype(int)

            mae = mean_absolute_error(y_true, y_pred)
            rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            acc = accuracy_score(y_true, y_pred)

            rows.append({
                "Model": m.name,
                "Accuracy": acc,
                "MAE": mae,
                "RMSE": rmse,
            })

        df_metrics = pd.DataFrame(rows).set_index("Model")
        return df_metrics

    # ---------- Plots ----------
    def plot_metrics_bar(self, save_dir=None, also_pdf=False):
        df = self.compute_metrics().reset_index()

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        sns.barplot(data=df, x="Model", y="Accuracy", ax=axes[0])
        axes[0].set_title("Accuracy")
        axes[0].set_ylim(0, 1)
        axes[0].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="MAE", ax=axes[1])
        axes[1].set_title("MAE")
        axes[1].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="RMSE", ax=axes[2])
        axes[2].set_title("RMSE")
        axes[2].grid(alpha=0.3, axis="y")

        for ax in axes:
            ax.set_xlabel("")
            for label in ax.get_xticklabels():
                label.set_rotation(20)
                label.set_ha("right")

        # plt.suptitle("Q-LLM — Metrics Comparison", y=1.05)
        plt.tight_layout()
        self._savefig(save_dir, "00_metrics_bar.png", also_pdf=also_pdf)
        plt.show()

    def plot_true_vs_pred_scatter(self, save_dir=None, also_pdf=False):
        name_to_model = {m.name: m for m in self.models}
        main_names = [n for n in ["LLM", "Q-LLM", "Quantum kNN", "Grover-only LLM"] if n in name_to_model]

        plt.figure(figsize=(10, 6))
        for name in main_names:
            m = name_to_model[name]
            y_true = np.asarray(m.y_true).astype(int)
            y_pred = np.asarray(m.y_pred).astype(int)
            plt.scatter(y_true, y_pred, alpha=0.45, s=45, label=name)

        plt.plot([1, 5], [1, 5], "k--", alpha=0.6)
        plt.xlim(0.5, 5.5)
        plt.ylim(0.5, 5.5)
        plt.xticks([1, 2, 3, 4, 5])
        plt.yticks([1, 2, 3, 4, 5])
        plt.xlabel("True Rating")
        plt.ylabel("Predicted Rating")
        plt.title("Q-LLM — True vs Predicted Ratings")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        self._savefig(save_dir, "01_true_vs_pred_scatter.png", also_pdf=also_pdf)
        plt.show()

    def plot_confusion_matrices(self, save_dir=None, also_pdf=False):
        name_to_model = {m.name: m for m in self.models}
        selected = [n for n in ["LLM", "Q-LLM"] if n in name_to_model]

        if len(selected) == 0:
            print("No LLM/Q-LLM models found for confusion matrices.")
            return

        fig, axes = plt.subplots(1, len(selected), figsize=(6 * len(selected), 5))
        if len(selected) == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            m = name_to_model[name]
            y_true = np.asarray(m.y_true).astype(int)
            y_pred = np.asarray(m.y_pred).astype(int)

            cm = confusion_matrix(y_true, y_pred, labels=self.labels)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=self.labels, yticklabels=self.labels, ax=ax)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_title(f"Confusion Matrix — {name}")

        plt.tight_layout()
        self._savefig(save_dir, "02_confusion_matrices.png", also_pdf=also_pdf)
        plt.show()

    def plot_error_histograms(self, save_dir=None, also_pdf=False):
        plt.figure(figsize=(10, 6))
        for m in self.models:
            y_true = np.asarray(m.y_true).astype(int)
            y_pred = np.asarray(m.y_pred).astype(int)
            errors = y_pred - y_true
            sns.histplot(errors, kde=True, stat="density", bins=np.arange(-4.5, 4.6, 1), label=m.name, alpha=0.35)

        plt.xlabel("Error = y_pred - y_true")
        plt.ylabel("Density")
        plt.title("Q-LLM — Error Distribution per Model")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        self._savefig(save_dir, "03_error_histograms.png", also_pdf=also_pdf)
        plt.show()

    def plot_abs_error_boxplot(self, save_dir=None, also_pdf=False):
        data = []
        for m in self.models:
            y_true = np.asarray(m.y_true).astype(int)
            y_pred = np.asarray(m.y_pred).astype(int)
            abs_err = np.abs(y_pred - y_true)
            data.append(pd.DataFrame({"Model": m.name, "Absolute Error": abs_err}))
        df = pd.concat(data, ignore_index=True)

        plt.figure(figsize=(9, 6))
        sns.boxplot(data=df, x="Model", y="Absolute Error")
        plt.title("Q-LLM — Absolute Error Distribution per Model")
        plt.grid(alpha=0.25, axis="y")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        self._savefig(save_dir, "04_abs_error_boxplot.png", also_pdf=also_pdf)
        plt.show()

    def plot_quantum_advantage_box(self, baseline_name="LLM", target_name="Q-LLM", save_dir=None, also_pdf=False):
        name_to_model = {m.name: m for m in self.models}
        if baseline_name not in name_to_model or target_name not in name_to_model:
            print(f"Missing {baseline_name} or {target_name} for advantage plot.")
            return

        m_base = name_to_model[baseline_name]
        m_tgt = name_to_model[target_name]

        y_true = np.asarray(m_base.y_true).astype(int)
        base_abs = np.abs(np.asarray(m_base.y_pred).astype(int) - y_true)
        tgt_abs = np.abs(np.asarray(m_tgt.y_pred).astype(int) - y_true)

        advantage = base_abs - tgt_abs  # >0 => Q-LLM better

        plt.figure(figsize=(7, 5))
        sns.boxplot(data=pd.DataFrame({"Quantum Advantage": advantage}))
        plt.axhline(0.0, color="red", linestyle="--", alpha=0.7)
        plt.title(f"Q-LLM — Advantage: |err_{baseline_name}| - |err_{target_name}|")
        plt.ylabel("Advantage (>0 means Q-LLM better)")
        plt.grid(alpha=0.25, axis="y")
        plt.tight_layout()
        self._savefig(save_dir, "05_quantum_advantage.png", also_pdf=also_pdf)
        plt.show()

    def plot_all(self, save_dir="figures_qllm", also_pdf=True):
        print("=== Metrics Table ===")
        display(self.compute_metrics())
        self.plot_metrics_bar(save_dir=save_dir, also_pdf=also_pdf)
        self.plot_true_vs_pred_scatter(save_dir=save_dir, also_pdf=also_pdf)
        self.plot_confusion_matrices(save_dir=save_dir, also_pdf=also_pdf)
        self.plot_error_histograms(save_dir=save_dir, also_pdf=also_pdf)
        self.plot_abs_error_boxplot(save_dir=save_dir, also_pdf=also_pdf)
        self.plot_quantum_advantage_box(
            baseline_name="LLM",
            target_name="Q-LLM",
            save_dir=save_dir,
            also_pdf=also_pdf
        )


# ----------------------------
# HOW TO RUN (exemple)
# ----------------------------
# Assure-toi que ces arrays existent déjà :
# true_ratings, llm_ratings, qknn_ratings, final_ratings, grover_ratings

models = [
    ModelResult("LLM",            true_ratings, llm_ratings),
    ModelResult("Quantum kNN",    true_ratings, qknn_ratings),
    ModelResult("Q-LLM",          true_ratings, final_ratings),
    ModelResult("Grover-only LLM",true_ratings, grover_ratings),
]

analyzer = RatingPredictionAnalyzer(models)

# Sauvegarde dans figures_qllm/ (PNG + PDF)
analyzer.plot_all(save_dir="figures_qllm", also_pdf=True)

print("\n📁 Figures saved in: figures_qllm/")


In [ ]:

idx = random.choice(indices)
review = df.loc[idx, TEXT_COL]
true_rating = int(df.loc[idx, RATING_COL])

print("ID review :", idx)
print("Vrai rating :", true_rating)
print("\nTARGET REVIEW :")
print(review[:1000])

out = hybrid_predict(review)

print("\n--- RÉSULTATS ---")
print("LLM rating           :", out["llm_rating"])
print("Quantum kNN (cont.)  :", out["quantum_knn_rating_cont"])
print("Quantum kNN (discret):", out["quantum_knn_rating"])
print("Final rating         :", out["final_rating"])
print("Poids fusion         :", out["weights"])

print("\nDétails LLM :")
print(json.dumps(out["llm_raw"], indent=2))

print("\nVoisins utilisés :")
for i, n in enumerate(out["neighbors"]):
    print(f"--- Neighbor {i+1} ---")
    print(f"index={n['index']}, rating={n['rating']}, similarity={n['similarity']:.3f}")
    print(n["text"][:200].replace("\n", " ") + "...")
